# 리랭킹 × 후보 수 × top-k 비교 (김연주)

검색 단계만 평가합니다 (답변 생성 API 호출 없음).

- **리랭킹**: lexical, cross-encoder (비교 기준: 리랭킹 없는 코사인 검색 none)
- **후보 수**: 30, 50, 70, 100 / **top-k**: 3, 5, 10
- **지표**: Recall@k(상위 k개에 나온 정답 문서 비율), MRR@k(첫 정답 문서 순위의 역수), 질문당 처리 시간
- 질문 임베딩은 `results/cache/query_embeddings.json` 캐시를 씁니다. 캐시에 없는 질문이 있으면 `ALLOW_API = True`로 바꿔야 API를 호출합니다.
- cross-encoder는 `torch`, `transformers`가 필요하고 GPU가 없으면 매우 느립니다. 첫 모델 로드 시간은 측정에서 뺍니다.
- 같은 코드를 터미널에서도 실행할 수 있습니다: `python experiments/rerank_grid_yjk.py --eval-file data/eval_sample_v0.json`

In [1]:
import sys
from pathlib import Path

# 노트북을 어디서 열든 저장소 루트를 찾습니다.
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "retrieval.py").is_file())
for path in (ROOT, ROOT / "experiments"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import rerank_grid_yjk as grid
from rerank_grid_pr_comment_yjk import build_comment

print("저장소:", ROOT.name)

저장소: rfp-rag


## 설정

In [2]:
EVAL_FILE = ROOT / "data/eval_sample_v0.json"
INDEX_DIR = ROOT / "indexes/parsing-v2-yjk"
MODES = ("lexical", "cross-encoder")
CANDIDATES = (30, 50, 70, 100)
TOP_KS = (3, 5, 10)
ALLOW_API = False  # 캐시에 없는 질문이 있을 때만 True (임베딩 API 비용 발생)

## 1. 평가셋·인덱스 준비, cross-encoder 모델 로드

In [3]:
cases = grid.load_cases(EVAL_FILE)
client, index, chunks, config, prep = grid.prepare(cases, INDEX_DIR, allow_api=ALLOW_API)
print(f"인덱스: 청크 {config['chunk_count']:,}개, {config['embedding_model']}")
load_ms = grid.warm_up(cases, client, index, chunks, config, MODES)

질문 14건 / 캐시에 없는 질문 0개
인덱스: 청크 9,789개, text-embedding-3-small
cross-encoder 모델 준비: 10181ms


## 2. 조합별 실행

순서: none(기준) → lexical → cross-encoder, 후보 30 → 100, top-k 3 → 10.

In [4]:
summary, records = grid.run_grid(cases, client, index, chunks, config, MODES, CANDIDATES, TOP_KS)

| 리랭킹 | 후보 수 | top-k | Recall@k | MRR@k | 질문당 평균(ms) | 중앙(ms) | p95(ms) | 전체(ms) |
| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| none | - | 3 | 0.976 | 0.929 | 11.5 | 10.9 | 12.6 | 161 |
| none | - | 5 | 0.976 | 0.929 | 11.1 | 10.5 | 11.2 | 155 |
| none | - | 10 | 1.000 | 0.929 | 11.1 | 10.5 | 10.9 | 155 |
| lexical | 30 | 3 | 0.952 | 1.000 | 21.4 | 20.9 | 21.8 | 300 |
| lexical | 30 | 5 | 0.976 | 1.000 | 21.1 | 20.6 | 20.9 | 296 |
| lexical | 30 | 10 | 1.000 | 1.000 | 21.2 | 20.7 | 22.3 | 297 |
| lexical | 50 | 3 | 0.976 | 1.000 | 28.0 | 27.3 | 33.2 | 393 |
| lexical | 50 | 5 | 0.976 | 1.000 | 27.7 | 27.0 | 28.1 | 387 |
| lexical | 50 | 10 | 1.000 | 1.000 | 27.6 | 27.2 | 27.8 | 387 |
| lexical | 70 | 3 | 0.952 | 1.000 | 34.5 | 33.9 | 36.8 | 484 |
| lexical | 70 | 5 | 0.976 | 1.000 | 35.1 | 33.7 | 37.6 | 492 |
| lexical | 70 | 10 | 1.000 | 1.000 | 34.4 | 33.8 | 34.7 | 482 |
| lexical | 100 | 3 | 0.976 | 1.000 | 44.9 | 44.4 | 45.7 | 628 |
| lexical | 100 | 5 | 0.976 

## 3. 결과 파일 저장

- 상세 결과 JSON: `results/rerank_grid_<측정 시각>.json` (Git 제외)
- 공유용 요약: `results/reports/rerank_grid_yjk_<평가셋>.md` (집계 지표만)

In [5]:
output, report, meta = grid.save_results(EVAL_FILE, INDEX_DIR, config, prep, load_ms, summary, records)
print("상세 결과:", output.relative_to(ROOT))
print("공유용 요약:", report.relative_to(ROOT))

상세 결과: results/rerank_grid_20261006T043614Z.json
공유용 요약: results/reports/rerank_grid_yjk_eval_sample_v0.md


## 4. PR 댓글 생성

수치에서 바로 나오는 사실만 씁니다. 후보 수 권장 같은 판단은 직접 덧붙이세요.

In [6]:
comment = build_comment({"settings": meta, "summary": summary}, {case.get("id"): case for case in cases})
pr_comment = ROOT / f"results/reports/pr_comment_rerank_grid_yjk_{EVAL_FILE.stem}.md"
pr_comment.write_text(comment, encoding="utf-8")
print(comment)
print("저장:", pr_comment.relative_to(ROOT))

## 추가 커밋

- `3dfbb90` merge: feature/parsing(청킹 연계용 파싱 개선)을 retrieval/max-per-doc에 병합
- `ba412a8` feat: 리랭킹 방식·후보 수·top-k 조합별 Recall@k·MRR@k·처리 시간 비교 실험 추가

## 리랭킹 × 후보 수 × top-k 비교: eval_sample_v0.json (14문항)

인덱스: 청크 9,789개 (1000/150, fixed), `text-embedding-3-small`, 측정 20261006T043614Z (UTC)

| 리랭킹 | 후보 수 | Recall@3 / @5 / @10 | MRR@3 / @5 / @10 | 질문당 평균 시간 |
| --- | ---: | --- | --- | ---: |
| 없음 | - | 0.976 / 0.976 / 1.000 | 0.929 / 0.929 / 0.929 | 11ms |
| lexical | 30 | 0.952 / 0.976 / 1.000 | 1.000 / 1.000 / 1.000 | 21ms |
| lexical | 50 | 0.976 / 0.976 / 1.000 | 1.000 / 1.000 / 1.000 | 28ms |
| lexical | 70 | 0.952 / 0.976 / 1.000 | 1.000 / 1.000 / 1.000 | 35ms |
| lexical | 100 | 0.976 / 0.976 / 0.976 | 1.000 / 1.000 / 1.000 | 45ms |
| cross-encoder | 30 | 0.976 / 1.000 / 1.000 | 1.000 / 1.000 / 1.000 | 475ms |
| cross-encoder | 50 | 0.976 / 0.976 / 1.000 | 1.000 / 1.000 / 1.000 | 823ms |
| cross-encoder | 70 | 0.976 / 0.976 / 1.000 | 1.000 / 1.000 / 1.000 | 1.17초 |
| cross-